<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_04_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 · Notebook 04 — assemble the report

**Paired with L9.1 · Laminar Flow**

Collects every run from notebooks 01–03 into a markdown report with the
configuration table, the error tables, and the questions you must answer.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · Load every run

In [ ]:
import pickle

runs = []
for f in ("nb01_baseline.pkl", "nb02_runs.pkl", "nb03_sweep.pkl"):
    path = os.path.join(pb.OUTPUT_DIR, f)
    if os.path.exists(path):
        with open(path, "rb") as fh:
            d = pickle.load(fh)
        runs.extend(d if isinstance(d, list) else [d])
    else:
        print(f"  missing: {path}  (run the notebook that writes it)")

print(f"{len(runs)} runs loaded")
for i, r in enumerate(runs, 1):
    print(f"  {i}. {r['config']}  ->  rel L2 (u) {r['mean_u_rel']:.3e}")

## 2 · Write it out

In [ ]:
path = pb.make_report(runs,
                      filename=os.path.join(pb.OUTPUT_DIR, "Ex09.1_report.md"),
                      author="YOUR NAME",
                      notes="Replace this with anything you want recorded.")

# On Colab, download it:
# from google.colab import files; files.download(path)
print(open(path).read()[:1500])

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex09.1_report.md into Ex09.1_report.pdf, with any figure
# saved as Ex09.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex09.1_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex09.1_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex09.1_report.pdf")
print("written Ex09.1_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex09.1_report.pdf")
except ImportError:
    pass


## 3 · Submitting

The generated file contains your configuration table and error tables, plus
five questions under **Your interpretation**. Answer each in a short paragraph
and submit the completed markdown together with the figures you consider
relevant.

The numbers are produced for you. The marks are for the interpretation.

## 4 · Extensions

- Add a variable-viscosity case: make `nu` a function of position and see what breaks.
- Expose `t_end` in the panel and study whether a longer window is harder.
- Replace the exact-solution boundary data with a coarse "measurement" set of
  30 scattered points and check whether the field is still recovered.
- Make `nu` trainable and identify it from the same 30 points. This is the
  inverse problem of slide 18, in miniature.